In [14]:
import json
import numpy as np
from confluent_kafka import Consumer
from pymongo import MongoClient

BOOTSTRAP = '172.27.110.36:9092'
TOPIC = 'coingecko_volumes'

def aggregate_to_4h(volume_data):
    if not isinstance(volume_data, list) or not volume_data:
        return [0.0] * 6
    volumes = []
    for point in volume_data:
        try:
            if isinstance(point, (list, tuple)) and len(point) >= 2:
                volumes.append(float(point[1]))
        except:
            continue
    if not volumes:
        return [0.0] * 6
    window_size = max(1, len(volumes) // 6)
    result = []
    for i in range(6):
        start = i * window_size
        end = len(volumes) if i == 5 else start + window_size
        window = volumes[start:end]
        result.append(float(np.mean(window)) if window else 0.0)
    return result

mongo = MongoClient("mongodb://localhost:27017/")
collection = mongo["coingecko"]["exchange_volumes"]

consumer = Consumer({
    'bootstrap.servers': BOOTSTRAP,
    'group.id': 'test_group_777',      # новое имя!
    'auto.offset.reset': 'earliest',
    'enable.auto.commit': True
})

consumer.subscribe([TOPIC])

print("Consumer запущен. Жду сообщения...\n")

try:
    while True:
        msg = consumer.poll(1.0)
        if msg is None:
            continue
        if msg.error():
            print("Ошибка:", msg.error())
            continue
        
        data = json.loads(msg.value().decode('utf-8'))
        exchange = list(data.keys())[0]
        volume_data = list(data.values())[0]
        
        volumes = aggregate_to_4h(volume_data)
        
        doc = {
            "exchange": exchange,
            "volume_1": volumes[0],
            "volume_2": volumes[1],
            "volume_3": volumes[2],
            "volume_4": volumes[3],
            "volume_5": volumes[4],
            "volume_6": volumes[5]
        }
        
        collection.update_one({"exchange": exchange}, {"$set": doc}, upsert=True)
        result = collection.update_one(
            {"exchange": exchange},
            {"$set": doc},
            upsert=True
        )
        
        print(f"→ MongoDB: {exchange} | matched: {result.matched_count}, upserted: {result.upserted_id is not None}")
        print(f"→ MongoDB: {exchange} | {[round(v, 4) for v in volumes]}")

except KeyboardInterrupt:
    print("\nОстановлен")
finally:
    consumer.close()

Consumer запущен. Жду сообщения...

→ MongoDB: ok | matched: 1, upserted: False
→ MongoDB: ok | [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
→ MongoDB: 10KSwap | matched: 1, upserted: False
→ MongoDB: 10KSwap | [0.0627, 0.0623, 0.057, 0.0579, 0.0714, 0.0791]
→ MongoDB: 9inch | matched: 1, upserted: False
→ MongoDB: 9inch | [0.545, 0.5305, 0.5478, 0.553, 0.6531, 0.6685]

Остановлен


In [ ]:
import requests
import urllib3
urllib3.disable_warnings()

exchange_id = "aevo"   # или любая другая
url = f"https://api.coingecko.com/api/v3/exchanges/{exchange_id}/volume_chart"
data = requests.get(url, params={"days": "1"}, verify=False).json()

print(data[:5])          # первые точки
print("Максимальный объём:", max([float(x[1]) for x in data]) if data else 0)

In [ ]:
from confluent_kafka import Consumer
c = Consumer({
    'bootstrap.servers': '172.27.110.36:9092',
    'group.id': 'ok1',
    'auto.offset.reset': 'earliest'
})
c.subscribe(['coingecko_volumes'])
print('waiting')
while True:
    m = c.poll(1.0)
    if m and not m.error():
        print(m.value())
        break
c.close()